# Artikate CV Take-Home — Colab GPU Run

**Before running:** Runtime → Change runtime type → **T4 GPU** (or any GPU) → Save.

This notebook clones the GitHub repo, fine-tunes YOLOv8n on the NEU-DET proxy set, exports ONNX, runs video inference + FP32/INT8 latency benchmark.

**ORT note:** Newest `onnxruntime-gpu` wants CUDA 13 (`libcudart.so.13`) which Colab T4 lacks. We use **CPU** `onnxruntime` for infer/benchmark; **training still uses the T4** via PyTorch.

In [ ]:
# 1) GPU check
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → GPU'

In [ ]:
# 2) Clone repo
import os
REPO = 'https://github.com/shivenswaroop/artikate-cv-takehome.git'
ROOT = '/content/artikate-cv-takehome'
if not os.path.exists(ROOT):
    !git clone --depth 1 {REPO} {ROOT}
else:
    %cd {ROOT}
    !git pull --ff-only || true
%cd {ROOT}
!pwd && ls -la

In [ ]:
# 3) Install deps
# Training uses Colab's torch+CUDA (GPU).
# Latest onnxruntime-gpu often wants CUDA 13 (libcudart.so.13) which Colab T4
# does not ship — use CPU onnxruntime for export/infer/benchmark instead.
%pip install -q -U ultralytics onnx "onnxruntime>=1.19,<1.23" opencv-python-headless PyYAML pandas tqdm Pillow pytest onnxslim
import os
os.environ['YOLO_CONFIG_DIR'] = '/content/ultralytics_config'
os.makedirs(os.environ['YOLO_CONFIG_DIR'], exist_ok=True)

# Drop a broken GPU ORT wheel if a previous cell installed one
import importlib, sys
for mod in list(sys.modules):
    if mod.startswith('onnxruntime'):
        del sys.modules[mod]

import ultralytics
import onnxruntime as ort
print('ultralytics', ultralytics.__version__)
print('ort', ort.__version__, 'providers', ort.get_available_providers())
print('Note: ORT is CPU here; YOLO training still uses the T4 GPU.')

In [ ]:
# 4) Ensure proxy dataset + held-out video exist
from pathlib import Path
ROOT = Path('/content/artikate-cv-takehome')
n_train = len(list((ROOT / 'data/proxy_neu/images/train').glob('*.jpg'))) if (ROOT / 'data/proxy_neu/images/train').exists() else 0
print('train images already in repo:', n_train)
if n_train < 100:
    !python scripts/prepare_neu_proxy.py
else:
    # regenerate video if missing
    if not (ROOT / 'data/proxy_neu/video/heldout.mp4').exists():
        !python scripts/prepare_neu_proxy.py
!ls data/proxy_neu/images/train | wc -l
!ls -lh data/proxy_neu/video/heldout.mp4
!cat configs/data.yaml

In [ ]:
# 5) Train YOLOv8n on GPU (faster than laptop CPU)
!python scripts/train.py \
  --data configs/data.yaml \
  --model yolov8n.pt \
  --epochs 40 \
  --batch 16 \
  --imgsz 640 \
  --device 0 \
  --workers 2 \
  --name neu_proxy_colab
!ls -lh weights/best.pt

In [ ]:
# 6) Export ONNX + INT8 + val metrics
!python scripts/export_onnx.py --weights weights/best.pt
!python scripts/quantize_int8.py
!python scripts/eval_val.py --weights weights/best.pt --data configs/data.yaml --device 0
!cat results/val_metrics.json

In [ ]:
# 7) Held-out video inference (first 90 frames) + FP32/INT8 latency
!python scripts/infer_video.py \
  --model weights/best.onnx \
  --video data/proxy_neu/video/heldout.mp4 \
  --max-frames 90 \
  --save-vis \
  --out-csv results/video_detections.csv
!python scripts/benchmark.py \
  --fp32 weights/best.onnx \
  --int8 weights/best_int8.onnx \
  --video data/proxy_neu/video/heldout.mp4 \
  --runs 30
!echo '--- latency ---' && cat results/video_latency_summary.txt
!echo '--- benchmark ---' && cat results/benchmark.json
!echo '--- sample dets ---' && head -10 results/video_detections.csv

In [ ]:
# 8) Show a few detection overlays
from pathlib import Path
from IPython.display import Image, display
frames = sorted(Path('results/frames').glob('frame_*.jpg'))[:4]
print(f'{len(list(Path("results/frames").glob("frame_*.jpg")))} frames saved; showing {len(frames)}')
for p in frames:
    print(p.name)
    display(Image(filename=str(p), width=360))

In [ ]:
# 9) Optional: zip artifacts to download
!mkdir -p /content/artifacts
!cp -f weights/best.pt weights/best.onnx weights/best_int8.onnx /content/artifacts/ 2>/dev/null || true
!cp -f results/val_metrics.json results/benchmark.json results/video_detections.csv results/video_latency_summary.txt /content/artifacts/ 2>/dev/null || true
!cd /content && zip -qr artikate_colab_artifacts.zip artifacts
from google.colab import files
print('Download artikate_colab_artifacts.zip from the next dialog (or Files sidebar).')
files.download('/content/artikate_colab_artifacts.zip')

## Notes for the assignment write-up

- Report **Colab GPU model** from `nvidia-smi` (e.g. T4) for training time.
- Latency from this notebook’s ORT path may use GPU providers — say so in the README; laptop CPU numbers stay valid as a second baseline.
- Paste your Loom recording after you screen-record this notebook run.